In [14]:
# ============================================================
# Seed naked ATM EOD panel from existing historical all-tenor source
# ============================================================

# Prefer the full recalibration universe if it has quotes/strike.
# Fall back to put_candidate_universe_pricing if needed.
seed_choice = None

preferred_files = [
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet",
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet",
]

for path in preferred_files:
    if not path.exists():
        continue

    temp = pd.read_parquet(path).copy()

    has_quotes = all(c in temp.columns for c in ["entry_bid", "entry_ask", "entry_mid"])
    has_strike = "short_strike_selected" in temp.columns
    has_tenor = any(c in temp.columns for c in ["entry_tenor", "tenor", "target_tenor"])
    has_date = any(c in temp.columns for c in ["trade_dt", "trade_date", "trade_date_ts"])

    if has_quotes and has_strike and has_tenor and has_date:
        seed_choice = path
        break

assert seed_choice is not None, (
    "Could not find a seed file with entry quotes, selected strike, date, and tenor. "
    "Inspect seed_inventory above."
)

seed_raw = pd.read_parquet(seed_choice).copy()

print("Using seed source:")
print(seed_choice)
print("Rows:", len(seed_raw))
print("Columns:", seed_raw.columns.tolist())

# Resolve source columns.
source_date_col = next(c for c in ["trade_dt", "trade_date", "trade_date_ts"] if c in seed_raw.columns)
source_tenor_col = next(c for c in ["entry_tenor", "tenor", "target_tenor"] if c in seed_raw.columns)

def first_existing_col(df, candidates, default_value=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default_value

def tenor_group_from_tenor(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

eod_seed = pd.DataFrame()

eod_seed["trade_dt"] = parse_project_date(seed_raw[source_date_col]).dt.normalize()
eod_seed["quote_timestamp"] = first_existing_col(seed_raw, ["quote_timestamp", "quote_time", "market_close_time"])
eod_seed["entry_tenor"] = seed_raw[source_tenor_col].astype(int)

eod_seed["expiry"] = parse_project_date(
    first_existing_col(seed_raw, ["selected_expiry_date", "target_expiry_date", "expiry"])
).dt.normalize()

eod_seed["actual_dte"] = first_existing_col(seed_raw, ["actual_dte"])
eod_seed["underlying_price"] = first_existing_col(seed_raw, ["entry_spx_close", "spx_close", "underlying_price"])
eod_seed["atm_strike"] = first_existing_col(seed_raw, ["short_strike_selected", "raw_short_strike", "atm_strike"])

eod_seed["atm_put_bid"] = first_existing_col(seed_raw, ["entry_bid", "atm_put_bid"])
eod_seed["atm_put_ask"] = first_existing_col(seed_raw, ["entry_ask", "atm_put_ask"])
eod_seed["atm_put_mid"] = first_existing_col(seed_raw, ["entry_mid", "atm_put_mid"])

eod_seed["atm_put_iv"] = first_existing_col(seed_raw, ["atm_put_iv", "vix_style_vol"])
eod_seed["atm_put_delta"] = first_existing_col(seed_raw, ["atm_put_delta", "put_delta"])

eod_seed["tenor_group"] = eod_seed["entry_tenor"].map(tenor_group_from_tenor)

# Keep useful research/live feature columns when present.
passthrough_cols = [
    "vix_style_vol",
    "implied_variance",
    "spx_rsi_14",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
    "pred_vol_pct_hybrid_back_har_only",
    "expiry_underlying_price",
    "expiry_spx_close",
    "expiry_intrinsic_value",
    "expired_otm",
    "entry_credit_points",
    "short_option_pnl_points",
    "normalized_pnl_dollars",
    "test_pnl",
    "win",
    "test_win",
    "pricing_status",
    "mapping_status",
    "selected_chain_file",
]

for c in passthrough_cols:
    if c in seed_raw.columns:
        eod_seed[c] = seed_raw[c]

# Keep only target tenors.
eod_seed = eod_seed[eod_seed["entry_tenor"].isin(TARGET_TENORS)].copy()

# Deduplicate one row per trade_dt / tenor.
eod_seed = (
    eod_seed
    .sort_values(["trade_dt", "entry_tenor"])
    .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
    .reset_index(drop=True)
)

# Basic validation.
expected_rows = eod_seed["trade_dt"].nunique() * len(TARGET_TENORS)
actual_rows = len(eod_seed)

print("EOD seed panel:")
print("Rows:", actual_rows)
print("Unique dates:", eod_seed["trade_dt"].nunique())
print("Date range:", eod_seed["trade_dt"].min(), "to", eod_seed["trade_dt"].max())
print("Unique tenors:", sorted(eod_seed["entry_tenor"].unique().tolist()))
print("Expected full grid rows:", expected_rows)
print("Missing grid rows:", expected_rows - actual_rows)

display(eod_seed.tail(20))

# Save as the new production EOD panel.
eod_seed.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

print("Saved seeded EOD panel:")
print(NAKED_ATM_EOD_PANEL)

Using seed source:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
Rows: 18099
Columns: ['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_ot

,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file
18079,2026-06-23,NaN,30,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.586777,NaN,back_27_33d,19.586777,0.038364,46.752076,46.752076,0.327861,-0.509629,-1.213937,-0.861783,0.313995,-0.596199,-1.266877,-1.266877,0.327861,-0.485334,-1.203989,-1.203989,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18080,2026-06-23,NaN,33,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.729159,NaN,back_27_33d,19.729159,0.038924,46.752076,46.752076,0.431584,-0.343885,-1.067693,-0.705789,0.377271,-0.512792,-1.208011,-1.208011,0.431584,-0.318612,-1.059780,-1.059780,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18081,2026-06-24,NaN,9,2026-07-02,8,7358.22,7355.0,69.8,70.9,70.35,19.326567,NaN,front_9_15d,19.326567,0.037352,46.342083,46.342083,0.567211,0.243603,-0.274788,-0.015593,0.669279,0.150559,-0.410467,-0.410467,0.567211,0.259376,-0.271507,-0.271507,NaN,NaN,None,69.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260702_160000.pkl
18082,2026-06-24,NaN,12,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.118185,NaN,front_9_15d,19.118185,0.036550,46.342083,46.342083,0.418108,-0.183775,-0.648595,-0.416185,0.739274,0.251112,-0.375552,-0.375552,0.418108,-0.162785,-0.644314,-0.644314,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18083,2026-06-24,NaN,15,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,18.992058,NaN,front_9_15d,18.992058,0.036070,46.342083,46.342083,0.068247,-0.766437,-1.282447,-1.024442,0.550836,-0.037813,-0.696963,-0.696963,0.068247,-0.740886,-1.271686,-1.271686,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18084,2026-06-24,NaN,18,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.091524,NaN,middle_18_24d,19.091524,0.036449,46.342083,46.342083,0.249885,-0.490428,-1.065457,-0.777942,0.621234,0.047563,-0.638722,-0.638722,0.249885,-0.466356,-1.057138,-1.057138,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18085,2026-06-24,NaN,21,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.240291,NaN,middle_18_24d,19.240291,0.037019,46.342083,46.342083,-0.002282,-1.133748,-1.627984,-1.380866,0.395771,-0.149625,-0.840562,-0.840562,-0.002282,-1.101711,-1.612012,-1.612012,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18086,2026-06-24,NaN,24,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.348982,NaN,middle_18_24d,19.348982,0.037438,46.342083,46.342083,0.113298,-0.899157,-1.475705,-1.187431,0.407037,-0.210791,-0.902639,-0.902639,0.113298,-0.873174,-1.462744,-1.462744,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18087,2026-06-24,NaN,27,2026-07-24,30,7358.22,7350.0,119.4,120.6,120.00,19.430528,NaN,back_27_33d,19.430528,0.037755,46.342083,46

Saved seeded EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet


In [15]:
# ============================================================
# Seed naked ATM EOD panel from existing historical all-tenor source
# ============================================================

# Prefer the full recalibration universe if it has quotes/strike.
# Fall back to put_candidate_universe_pricing if needed.
seed_choice = None

preferred_files = [
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet",
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet",
]

for path in preferred_files:
    if not path.exists():
        continue

    temp = pd.read_parquet(path).copy()

    has_quotes = all(c in temp.columns for c in ["entry_bid", "entry_ask", "entry_mid"])
    has_strike = "short_strike_selected" in temp.columns
    has_tenor = any(c in temp.columns for c in ["entry_tenor", "tenor", "target_tenor"])
    has_date = any(c in temp.columns for c in ["trade_dt", "trade_date", "trade_date_ts"])

    if has_quotes and has_strike and has_tenor and has_date:
        seed_choice = path
        break

assert seed_choice is not None, (
    "Could not find a seed file with entry quotes, selected strike, date, and tenor. "
    "Inspect seed_inventory above."
)

seed_raw = pd.read_parquet(seed_choice).copy()

print("Using seed source:")
print(seed_choice)
print("Rows:", len(seed_raw))
print("Columns:", seed_raw.columns.tolist())

# Resolve source columns.
source_date_col = next(c for c in ["trade_dt", "trade_date", "trade_date_ts"] if c in seed_raw.columns)
source_tenor_col = next(c for c in ["entry_tenor", "tenor", "target_tenor"] if c in seed_raw.columns)

def first_existing_col(df, candidates, default_value=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default_value

def tenor_group_from_tenor(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

eod_seed = pd.DataFrame()

eod_seed["trade_dt"] = parse_project_date(seed_raw[source_date_col]).dt.normalize()
eod_seed["quote_timestamp"] = first_existing_col(seed_raw, ["quote_timestamp", "quote_time", "market_close_time"])
eod_seed["entry_tenor"] = seed_raw[source_tenor_col].astype(int)

eod_seed["expiry"] = parse_project_date(
    first_existing_col(seed_raw, ["selected_expiry_date", "target_expiry_date", "expiry"])
).dt.normalize()

eod_seed["actual_dte"] = first_existing_col(seed_raw, ["actual_dte"])
eod_seed["underlying_price"] = first_existing_col(seed_raw, ["entry_spx_close", "spx_close", "underlying_price"])
eod_seed["atm_strike"] = first_existing_col(seed_raw, ["short_strike_selected", "raw_short_strike", "atm_strike"])

eod_seed["atm_put_bid"] = first_existing_col(seed_raw, ["entry_bid", "atm_put_bid"])
eod_seed["atm_put_ask"] = first_existing_col(seed_raw, ["entry_ask", "atm_put_ask"])
eod_seed["atm_put_mid"] = first_existing_col(seed_raw, ["entry_mid", "atm_put_mid"])

eod_seed["atm_put_iv"] = first_existing_col(seed_raw, ["atm_put_iv", "vix_style_vol"])
eod_seed["atm_put_delta"] = first_existing_col(seed_raw, ["atm_put_delta", "put_delta"])

eod_seed["tenor_group"] = eod_seed["entry_tenor"].map(tenor_group_from_tenor)

# Keep useful research/live feature columns when present.
passthrough_cols = [
    "vix_style_vol",
    "implied_variance",
    "spx_rsi_14",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
    "pred_vol_pct_hybrid_back_har_only",
    "expiry_underlying_price",
    "expiry_spx_close",
    "expiry_intrinsic_value",
    "expired_otm",
    "entry_credit_points",
    "short_option_pnl_points",
    "normalized_pnl_dollars",
    "test_pnl",
    "win",
    "test_win",
    "pricing_status",
    "mapping_status",
    "selected_chain_file",
]

for c in passthrough_cols:
    if c in seed_raw.columns:
        eod_seed[c] = seed_raw[c]

# Keep only target tenors.
eod_seed = eod_seed[eod_seed["entry_tenor"].isin(TARGET_TENORS)].copy()

# Deduplicate one row per trade_dt / tenor.
eod_seed = (
    eod_seed
    .sort_values(["trade_dt", "entry_tenor"])
    .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
    .reset_index(drop=True)
)

# Basic validation.
expected_rows = eod_seed["trade_dt"].nunique() * len(TARGET_TENORS)
actual_rows = len(eod_seed)

print("EOD seed panel:")
print("Rows:", actual_rows)
print("Unique dates:", eod_seed["trade_dt"].nunique())
print("Date range:", eod_seed["trade_dt"].min(), "to", eod_seed["trade_dt"].max())
print("Unique tenors:", sorted(eod_seed["entry_tenor"].unique().tolist()))
print("Expected full grid rows:", expected_rows)
print("Missing grid rows:", expected_rows - actual_rows)

display(eod_seed.tail(20))

# Save as the new production EOD panel.
eod_seed.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

print("Saved seeded EOD panel:")
print(NAKED_ATM_EOD_PANEL)

Using seed source:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
Rows: 18099
Columns: ['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_ot

,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file
18079,2026-06-23,NaN,30,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.586777,NaN,back_27_33d,19.586777,0.038364,46.752076,46.752076,0.327861,-0.509629,-1.213937,-0.861783,0.313995,-0.596199,-1.266877,-1.266877,0.327861,-0.485334,-1.203989,-1.203989,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18080,2026-06-23,NaN,33,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.729159,NaN,back_27_33d,19.729159,0.038924,46.752076,46.752076,0.431584,-0.343885,-1.067693,-0.705789,0.377271,-0.512792,-1.208011,-1.208011,0.431584,-0.318612,-1.059780,-1.059780,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18081,2026-06-24,NaN,9,2026-07-02,8,7358.22,7355.0,69.8,70.9,70.35,19.326567,NaN,front_9_15d,19.326567,0.037352,46.342083,46.342083,0.567211,0.243603,-0.274788,-0.015593,0.669279,0.150559,-0.410467,-0.410467,0.567211,0.259376,-0.271507,-0.271507,NaN,NaN,None,69.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260702_160000.pkl
18082,2026-06-24,NaN,12,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.118185,NaN,front_9_15d,19.118185,0.036550,46.342083,46.342083,0.418108,-0.183775,-0.648595,-0.416185,0.739274,0.251112,-0.375552,-0.375552,0.418108,-0.162785,-0.644314,-0.644314,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18083,2026-06-24,NaN,15,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,18.992058,NaN,front_9_15d,18.992058,0.036070,46.342083,46.342083,0.068247,-0.766437,-1.282447,-1.024442,0.550836,-0.037813,-0.696963,-0.696963,0.068247,-0.740886,-1.271686,-1.271686,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18084,2026-06-24,NaN,18,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.091524,NaN,middle_18_24d,19.091524,0.036449,46.342083,46.342083,0.249885,-0.490428,-1.065457,-0.777942,0.621234,0.047563,-0.638722,-0.638722,0.249885,-0.466356,-1.057138,-1.057138,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18085,2026-06-24,NaN,21,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.240291,NaN,middle_18_24d,19.240291,0.037019,46.342083,46.342083,-0.002282,-1.133748,-1.627984,-1.380866,0.395771,-0.149625,-0.840562,-0.840562,-0.002282,-1.101711,-1.612012,-1.612012,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18086,2026-06-24,NaN,24,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.348982,NaN,middle_18_24d,19.348982,0.037438,46.342083,46.342083,0.113298,-0.899157,-1.475705,-1.187431,0.407037,-0.210791,-0.902639,-0.902639,0.113298,-0.873174,-1.462744,-1.462744,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18087,2026-06-24,NaN,27,2026-07-24,30,7358.22,7350.0,119.4,120.6,120.00,19.430528,NaN,back_27_33d,19.430528,0.037755,46.342083,46

Saved seeded EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet


In [16]:
# ============================================================
# Seed naked ATM EOD panel from existing historical all-tenor source
# ============================================================

# Prefer the full recalibration universe if it has quotes/strike.
# Fall back to put_candidate_universe_pricing if needed.
seed_choice = None

preferred_files = [
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet",
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet",
]

for path in preferred_files:
    if not path.exists():
        continue

    temp = pd.read_parquet(path).copy()

    has_quotes = all(c in temp.columns for c in ["entry_bid", "entry_ask", "entry_mid"])
    has_strike = "short_strike_selected" in temp.columns
    has_tenor = any(c in temp.columns for c in ["entry_tenor", "tenor", "target_tenor"])
    has_date = any(c in temp.columns for c in ["trade_dt", "trade_date", "trade_date_ts"])

    if has_quotes and has_strike and has_tenor and has_date:
        seed_choice = path
        break

assert seed_choice is not None, (
    "Could not find a seed file with entry quotes, selected strike, date, and tenor. "
    "Inspect seed_inventory above."
)

seed_raw = pd.read_parquet(seed_choice).copy()

print("Using seed source:")
print(seed_choice)
print("Rows:", len(seed_raw))
print("Columns:", seed_raw.columns.tolist())

# Resolve source columns.
source_date_col = next(c for c in ["trade_dt", "trade_date", "trade_date_ts"] if c in seed_raw.columns)
source_tenor_col = next(c for c in ["entry_tenor", "tenor", "target_tenor"] if c in seed_raw.columns)

def first_existing_col(df, candidates, default_value=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default_value

def tenor_group_from_tenor(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

eod_seed = pd.DataFrame()

eod_seed["trade_dt"] = parse_project_date(seed_raw[source_date_col]).dt.normalize()
eod_seed["quote_timestamp"] = first_existing_col(seed_raw, ["quote_timestamp", "quote_time", "market_close_time"])
eod_seed["entry_tenor"] = seed_raw[source_tenor_col].astype(int)

eod_seed["expiry"] = parse_project_date(
    first_existing_col(seed_raw, ["selected_expiry_date", "target_expiry_date", "expiry"])
).dt.normalize()

eod_seed["actual_dte"] = first_existing_col(seed_raw, ["actual_dte"])
eod_seed["underlying_price"] = first_existing_col(seed_raw, ["entry_spx_close", "spx_close", "underlying_price"])
eod_seed["atm_strike"] = first_existing_col(seed_raw, ["short_strike_selected", "raw_short_strike", "atm_strike"])

eod_seed["atm_put_bid"] = first_existing_col(seed_raw, ["entry_bid", "atm_put_bid"])
eod_seed["atm_put_ask"] = first_existing_col(seed_raw, ["entry_ask", "atm_put_ask"])
eod_seed["atm_put_mid"] = first_existing_col(seed_raw, ["entry_mid", "atm_put_mid"])

eod_seed["atm_put_iv"] = first_existing_col(seed_raw, ["atm_put_iv", "vix_style_vol"])
eod_seed["atm_put_delta"] = first_existing_col(seed_raw, ["atm_put_delta", "put_delta"])

eod_seed["tenor_group"] = eod_seed["entry_tenor"].map(tenor_group_from_tenor)

# Keep useful research/live feature columns when present.
passthrough_cols = [
    "vix_style_vol",
    "implied_variance",
    "spx_rsi_14",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
    "pred_vol_pct_hybrid_back_har_only",
    "expiry_underlying_price",
    "expiry_spx_close",
    "expiry_intrinsic_value",
    "expired_otm",
    "entry_credit_points",
    "short_option_pnl_points",
    "normalized_pnl_dollars",
    "test_pnl",
    "win",
    "test_win",
    "pricing_status",
    "mapping_status",
    "selected_chain_file",
]

for c in passthrough_cols:
    if c in seed_raw.columns:
        eod_seed[c] = seed_raw[c]

# Keep only target tenors.
eod_seed = eod_seed[eod_seed["entry_tenor"].isin(TARGET_TENORS)].copy()

# Deduplicate one row per trade_dt / tenor.
eod_seed = (
    eod_seed
    .sort_values(["trade_dt", "entry_tenor"])
    .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
    .reset_index(drop=True)
)

# Basic validation.
expected_rows = eod_seed["trade_dt"].nunique() * len(TARGET_TENORS)
actual_rows = len(eod_seed)

print("EOD seed panel:")
print("Rows:", actual_rows)
print("Unique dates:", eod_seed["trade_dt"].nunique())
print("Date range:", eod_seed["trade_dt"].min(), "to", eod_seed["trade_dt"].max())
print("Unique tenors:", sorted(eod_seed["entry_tenor"].unique().tolist()))
print("Expected full grid rows:", expected_rows)
print("Missing grid rows:", expected_rows - actual_rows)

display(eod_seed.tail(20))

# Save as the new production EOD panel.
eod_seed.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

print("Saved seeded EOD panel:")
print(NAKED_ATM_EOD_PANEL)

Using seed source:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
Rows: 18099
Columns: ['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_ot

,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file
18079,2026-06-23,NaN,30,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.586777,NaN,back_27_33d,19.586777,0.038364,46.752076,46.752076,0.327861,-0.509629,-1.213937,-0.861783,0.313995,-0.596199,-1.266877,-1.266877,0.327861,-0.485334,-1.203989,-1.203989,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18080,2026-06-23,NaN,33,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.729159,NaN,back_27_33d,19.729159,0.038924,46.752076,46.752076,0.431584,-0.343885,-1.067693,-0.705789,0.377271,-0.512792,-1.208011,-1.208011,0.431584,-0.318612,-1.059780,-1.059780,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18081,2026-06-24,NaN,9,2026-07-02,8,7358.22,7355.0,69.8,70.9,70.35,19.326567,NaN,front_9_15d,19.326567,0.037352,46.342083,46.342083,0.567211,0.243603,-0.274788,-0.015593,0.669279,0.150559,-0.410467,-0.410467,0.567211,0.259376,-0.271507,-0.271507,NaN,NaN,None,69.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260702_160000.pkl
18082,2026-06-24,NaN,12,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.118185,NaN,front_9_15d,19.118185,0.036550,46.342083,46.342083,0.418108,-0.183775,-0.648595,-0.416185,0.739274,0.251112,-0.375552,-0.375552,0.418108,-0.162785,-0.644314,-0.644314,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18083,2026-06-24,NaN,15,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,18.992058,NaN,front_9_15d,18.992058,0.036070,46.342083,46.342083,0.068247,-0.766437,-1.282447,-1.024442,0.550836,-0.037813,-0.696963,-0.696963,0.068247,-0.740886,-1.271686,-1.271686,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18084,2026-06-24,NaN,18,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.091524,NaN,middle_18_24d,19.091524,0.036449,46.342083,46.342083,0.249885,-0.490428,-1.065457,-0.777942,0.621234,0.047563,-0.638722,-0.638722,0.249885,-0.466356,-1.057138,-1.057138,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18085,2026-06-24,NaN,21,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.240291,NaN,middle_18_24d,19.240291,0.037019,46.342083,46.342083,-0.002282,-1.133748,-1.627984,-1.380866,0.395771,-0.149625,-0.840562,-0.840562,-0.002282,-1.101711,-1.612012,-1.612012,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18086,2026-06-24,NaN,24,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.348982,NaN,middle_18_24d,19.348982,0.037438,46.342083,46.342083,0.113298,-0.899157,-1.475705,-1.187431,0.407037,-0.210791,-0.902639,-0.902639,0.113298,-0.873174,-1.462744,-1.462744,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18087,2026-06-24,NaN,27,2026-07-24,30,7358.22,7350.0,119.4,120.6,120.00,19.430528,NaN,back_27_33d,19.430528,0.037755,46.342083,46

Saved seeded EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet


In [17]:
# ============================================================
# Seed naked ATM EOD panel from existing historical all-tenor source
# ============================================================

# Prefer the full recalibration universe if it has quotes/strike.
# Fall back to put_candidate_universe_pricing if needed.
seed_choice = None

preferred_files = [
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet",
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet",
]

for path in preferred_files:
    if not path.exists():
        continue

    temp = pd.read_parquet(path).copy()

    has_quotes = all(c in temp.columns for c in ["entry_bid", "entry_ask", "entry_mid"])
    has_strike = "short_strike_selected" in temp.columns
    has_tenor = any(c in temp.columns for c in ["entry_tenor", "tenor", "target_tenor"])
    has_date = any(c in temp.columns for c in ["trade_dt", "trade_date", "trade_date_ts"])

    if has_quotes and has_strike and has_tenor and has_date:
        seed_choice = path
        break

assert seed_choice is not None, (
    "Could not find a seed file with entry quotes, selected strike, date, and tenor. "
    "Inspect seed_inventory above."
)

seed_raw = pd.read_parquet(seed_choice).copy()

print("Using seed source:")
print(seed_choice)
print("Rows:", len(seed_raw))
print("Columns:", seed_raw.columns.tolist())

# Resolve source columns.
source_date_col = next(c for c in ["trade_dt", "trade_date", "trade_date_ts"] if c in seed_raw.columns)
source_tenor_col = next(c for c in ["entry_tenor", "tenor", "target_tenor"] if c in seed_raw.columns)

def first_existing_col(df, candidates, default_value=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default_value

def tenor_group_from_tenor(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

eod_seed = pd.DataFrame()

eod_seed["trade_dt"] = parse_project_date(seed_raw[source_date_col]).dt.normalize()
eod_seed["quote_timestamp"] = first_existing_col(seed_raw, ["quote_timestamp", "quote_time", "market_close_time"])
eod_seed["entry_tenor"] = seed_raw[source_tenor_col].astype(int)

eod_seed["expiry"] = parse_project_date(
    first_existing_col(seed_raw, ["selected_expiry_date", "target_expiry_date", "expiry"])
).dt.normalize()

eod_seed["actual_dte"] = first_existing_col(seed_raw, ["actual_dte"])
eod_seed["underlying_price"] = first_existing_col(seed_raw, ["entry_spx_close", "spx_close", "underlying_price"])
eod_seed["atm_strike"] = first_existing_col(seed_raw, ["short_strike_selected", "raw_short_strike", "atm_strike"])

eod_seed["atm_put_bid"] = first_existing_col(seed_raw, ["entry_bid", "atm_put_bid"])
eod_seed["atm_put_ask"] = first_existing_col(seed_raw, ["entry_ask", "atm_put_ask"])
eod_seed["atm_put_mid"] = first_existing_col(seed_raw, ["entry_mid", "atm_put_mid"])

eod_seed["atm_put_iv"] = first_existing_col(seed_raw, ["atm_put_iv", "vix_style_vol"])
eod_seed["atm_put_delta"] = first_existing_col(seed_raw, ["atm_put_delta", "put_delta"])

eod_seed["tenor_group"] = eod_seed["entry_tenor"].map(tenor_group_from_tenor)

# Keep useful research/live feature columns when present.
passthrough_cols = [
    "vix_style_vol",
    "implied_variance",
    "spx_rsi_14",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
    "pred_vol_pct_hybrid_back_har_only",
    "expiry_underlying_price",
    "expiry_spx_close",
    "expiry_intrinsic_value",
    "expired_otm",
    "entry_credit_points",
    "short_option_pnl_points",
    "normalized_pnl_dollars",
    "test_pnl",
    "win",
    "test_win",
    "pricing_status",
    "mapping_status",
    "selected_chain_file",
]

for c in passthrough_cols:
    if c in seed_raw.columns:
        eod_seed[c] = seed_raw[c]

# Keep only target tenors.
eod_seed = eod_seed[eod_seed["entry_tenor"].isin(TARGET_TENORS)].copy()

# Deduplicate one row per trade_dt / tenor.
eod_seed = (
    eod_seed
    .sort_values(["trade_dt", "entry_tenor"])
    .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
    .reset_index(drop=True)
)

# Basic validation.
expected_rows = eod_seed["trade_dt"].nunique() * len(TARGET_TENORS)
actual_rows = len(eod_seed)

print("EOD seed panel:")
print("Rows:", actual_rows)
print("Unique dates:", eod_seed["trade_dt"].nunique())
print("Date range:", eod_seed["trade_dt"].min(), "to", eod_seed["trade_dt"].max())
print("Unique tenors:", sorted(eod_seed["entry_tenor"].unique().tolist()))
print("Expected full grid rows:", expected_rows)
print("Missing grid rows:", expected_rows - actual_rows)

display(eod_seed.tail(20))

# Save as the new production EOD panel.
eod_seed.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

print("Saved seeded EOD panel:")
print(NAKED_ATM_EOD_PANEL)

Using seed source:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
Rows: 18099
Columns: ['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_ot

,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file
18079,2026-06-23,NaN,30,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.586777,NaN,back_27_33d,19.586777,0.038364,46.752076,46.752076,0.327861,-0.509629,-1.213937,-0.861783,0.313995,-0.596199,-1.266877,-1.266877,0.327861,-0.485334,-1.203989,-1.203989,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18080,2026-06-23,NaN,33,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.729159,NaN,back_27_33d,19.729159,0.038924,46.752076,46.752076,0.431584,-0.343885,-1.067693,-0.705789,0.377271,-0.512792,-1.208011,-1.208011,0.431584,-0.318612,-1.059780,-1.059780,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18081,2026-06-24,NaN,9,2026-07-02,8,7358.22,7355.0,69.8,70.9,70.35,19.326567,NaN,front_9_15d,19.326567,0.037352,46.342083,46.342083,0.567211,0.243603,-0.274788,-0.015593,0.669279,0.150559,-0.410467,-0.410467,0.567211,0.259376,-0.271507,-0.271507,NaN,NaN,None,69.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260702_160000.pkl
18082,2026-06-24,NaN,12,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.118185,NaN,front_9_15d,19.118185,0.036550,46.342083,46.342083,0.418108,-0.183775,-0.648595,-0.416185,0.739274,0.251112,-0.375552,-0.375552,0.418108,-0.162785,-0.644314,-0.644314,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18083,2026-06-24,NaN,15,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,18.992058,NaN,front_9_15d,18.992058,0.036070,46.342083,46.342083,0.068247,-0.766437,-1.282447,-1.024442,0.550836,-0.037813,-0.696963,-0.696963,0.068247,-0.740886,-1.271686,-1.271686,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18084,2026-06-24,NaN,18,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.091524,NaN,middle_18_24d,19.091524,0.036449,46.342083,46.342083,0.249885,-0.490428,-1.065457,-0.777942,0.621234,0.047563,-0.638722,-0.638722,0.249885,-0.466356,-1.057138,-1.057138,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18085,2026-06-24,NaN,21,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.240291,NaN,middle_18_24d,19.240291,0.037019,46.342083,46.342083,-0.002282,-1.133748,-1.627984,-1.380866,0.395771,-0.149625,-0.840562,-0.840562,-0.002282,-1.101711,-1.612012,-1.612012,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18086,2026-06-24,NaN,24,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.348982,NaN,middle_18_24d,19.348982,0.037438,46.342083,46.342083,0.113298,-0.899157,-1.475705,-1.187431,0.407037,-0.210791,-0.902639,-0.902639,0.113298,-0.873174,-1.462744,-1.462744,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18087,2026-06-24,NaN,27,2026-07-24,30,7358.22,7350.0,119.4,120.6,120.00,19.430528,NaN,back_27_33d,19.430528,0.037755,46.342083,46

Saved seeded EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet


In [18]:
# ============================================================
# Seed naked ATM EOD panel from existing historical all-tenor source
# ============================================================

# Prefer the full recalibration universe if it has quotes/strike.
# Fall back to put_candidate_universe_pricing if needed.
seed_choice = None

preferred_files = [
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet",
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet",
]

for path in preferred_files:
    if not path.exists():
        continue

    temp = pd.read_parquet(path).copy()

    has_quotes = all(c in temp.columns for c in ["entry_bid", "entry_ask", "entry_mid"])
    has_strike = "short_strike_selected" in temp.columns
    has_tenor = any(c in temp.columns for c in ["entry_tenor", "tenor", "target_tenor"])
    has_date = any(c in temp.columns for c in ["trade_dt", "trade_date", "trade_date_ts"])

    if has_quotes and has_strike and has_tenor and has_date:
        seed_choice = path
        break

assert seed_choice is not None, (
    "Could not find a seed file with entry quotes, selected strike, date, and tenor. "
    "Inspect seed_inventory above."
)

seed_raw = pd.read_parquet(seed_choice).copy()

print("Using seed source:")
print(seed_choice)
print("Rows:", len(seed_raw))
print("Columns:", seed_raw.columns.tolist())

# Resolve source columns.
source_date_col = next(c for c in ["trade_dt", "trade_date", "trade_date_ts"] if c in seed_raw.columns)
source_tenor_col = next(c for c in ["entry_tenor", "tenor", "target_tenor"] if c in seed_raw.columns)

def first_existing_col(df, candidates, default_value=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default_value

def tenor_group_from_tenor(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

eod_seed = pd.DataFrame()

eod_seed["trade_dt"] = parse_project_date(seed_raw[source_date_col]).dt.normalize()
eod_seed["quote_timestamp"] = first_existing_col(seed_raw, ["quote_timestamp", "quote_time", "market_close_time"])
eod_seed["entry_tenor"] = seed_raw[source_tenor_col].astype(int)

eod_seed["expiry"] = parse_project_date(
    first_existing_col(seed_raw, ["selected_expiry_date", "target_expiry_date", "expiry"])
).dt.normalize()

eod_seed["actual_dte"] = first_existing_col(seed_raw, ["actual_dte"])
eod_seed["underlying_price"] = first_existing_col(seed_raw, ["entry_spx_close", "spx_close", "underlying_price"])
eod_seed["atm_strike"] = first_existing_col(seed_raw, ["short_strike_selected", "raw_short_strike", "atm_strike"])

eod_seed["atm_put_bid"] = first_existing_col(seed_raw, ["entry_bid", "atm_put_bid"])
eod_seed["atm_put_ask"] = first_existing_col(seed_raw, ["entry_ask", "atm_put_ask"])
eod_seed["atm_put_mid"] = first_existing_col(seed_raw, ["entry_mid", "atm_put_mid"])

eod_seed["atm_put_iv"] = first_existing_col(seed_raw, ["atm_put_iv", "vix_style_vol"])
eod_seed["atm_put_delta"] = first_existing_col(seed_raw, ["atm_put_delta", "put_delta"])

eod_seed["tenor_group"] = eod_seed["entry_tenor"].map(tenor_group_from_tenor)

# Keep useful research/live feature columns when present.
passthrough_cols = [
    "vix_style_vol",
    "implied_variance",
    "spx_rsi_14",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
    "pred_vol_pct_hybrid_back_har_only",
    "expiry_underlying_price",
    "expiry_spx_close",
    "expiry_intrinsic_value",
    "expired_otm",
    "entry_credit_points",
    "short_option_pnl_points",
    "normalized_pnl_dollars",
    "test_pnl",
    "win",
    "test_win",
    "pricing_status",
    "mapping_status",
    "selected_chain_file",
]

for c in passthrough_cols:
    if c in seed_raw.columns:
        eod_seed[c] = seed_raw[c]

# Keep only target tenors.
eod_seed = eod_seed[eod_seed["entry_tenor"].isin(TARGET_TENORS)].copy()

# Deduplicate one row per trade_dt / tenor.
eod_seed = (
    eod_seed
    .sort_values(["trade_dt", "entry_tenor"])
    .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
    .reset_index(drop=True)
)

# Basic validation.
expected_rows = eod_seed["trade_dt"].nunique() * len(TARGET_TENORS)
actual_rows = len(eod_seed)

print("EOD seed panel:")
print("Rows:", actual_rows)
print("Unique dates:", eod_seed["trade_dt"].nunique())
print("Date range:", eod_seed["trade_dt"].min(), "to", eod_seed["trade_dt"].max())
print("Unique tenors:", sorted(eod_seed["entry_tenor"].unique().tolist()))
print("Expected full grid rows:", expected_rows)
print("Missing grid rows:", expected_rows - actual_rows)

display(eod_seed.tail(20))

# Save as the new production EOD panel.
eod_seed.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

print("Saved seeded EOD panel:")
print(NAKED_ATM_EOD_PANEL)

Using seed source:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
Rows: 18099
Columns: ['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_ot

,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file
18079,2026-06-23,NaN,30,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.586777,NaN,back_27_33d,19.586777,0.038364,46.752076,46.752076,0.327861,-0.509629,-1.213937,-0.861783,0.313995,-0.596199,-1.266877,-1.266877,0.327861,-0.485334,-1.203989,-1.203989,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18080,2026-06-23,NaN,33,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.729159,NaN,back_27_33d,19.729159,0.038924,46.752076,46.752076,0.431584,-0.343885,-1.067693,-0.705789,0.377271,-0.512792,-1.208011,-1.208011,0.431584,-0.318612,-1.059780,-1.059780,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18081,2026-06-24,NaN,9,2026-07-02,8,7358.22,7355.0,69.8,70.9,70.35,19.326567,NaN,front_9_15d,19.326567,0.037352,46.342083,46.342083,0.567211,0.243603,-0.274788,-0.015593,0.669279,0.150559,-0.410467,-0.410467,0.567211,0.259376,-0.271507,-0.271507,NaN,NaN,None,69.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260702_160000.pkl
18082,2026-06-24,NaN,12,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.118185,NaN,front_9_15d,19.118185,0.036550,46.342083,46.342083,0.418108,-0.183775,-0.648595,-0.416185,0.739274,0.251112,-0.375552,-0.375552,0.418108,-0.162785,-0.644314,-0.644314,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18083,2026-06-24,NaN,15,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,18.992058,NaN,front_9_15d,18.992058,0.036070,46.342083,46.342083,0.068247,-0.766437,-1.282447,-1.024442,0.550836,-0.037813,-0.696963,-0.696963,0.068247,-0.740886,-1.271686,-1.271686,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18084,2026-06-24,NaN,18,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.091524,NaN,middle_18_24d,19.091524,0.036449,46.342083,46.342083,0.249885,-0.490428,-1.065457,-0.777942,0.621234,0.047563,-0.638722,-0.638722,0.249885,-0.466356,-1.057138,-1.057138,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18085,2026-06-24,NaN,21,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.240291,NaN,middle_18_24d,19.240291,0.037019,46.342083,46.342083,-0.002282,-1.133748,-1.627984,-1.380866,0.395771,-0.149625,-0.840562,-0.840562,-0.002282,-1.101711,-1.612012,-1.612012,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18086,2026-06-24,NaN,24,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.348982,NaN,middle_18_24d,19.348982,0.037438,46.342083,46.342083,0.113298,-0.899157,-1.475705,-1.187431,0.407037,-0.210791,-0.902639,-0.902639,0.113298,-0.873174,-1.462744,-1.462744,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18087,2026-06-24,NaN,27,2026-07-24,30,7358.22,7350.0,119.4,120.6,120.00,19.430528,NaN,back_27_33d,19.430528,0.037755,46.342083,46

Saved seeded EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet


In [19]:
# ============================================================
# Seed naked ATM EOD panel from existing historical all-tenor source
# ============================================================

# Prefer the full recalibration universe if it has quotes/strike.
# Fall back to put_candidate_universe_pricing if needed.
seed_choice = None

preferred_files = [
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet",
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet",
]

for path in preferred_files:
    if not path.exists():
        continue

    temp = pd.read_parquet(path).copy()

    has_quotes = all(c in temp.columns for c in ["entry_bid", "entry_ask", "entry_mid"])
    has_strike = "short_strike_selected" in temp.columns
    has_tenor = any(c in temp.columns for c in ["entry_tenor", "tenor", "target_tenor"])
    has_date = any(c in temp.columns for c in ["trade_dt", "trade_date", "trade_date_ts"])

    if has_quotes and has_strike and has_tenor and has_date:
        seed_choice = path
        break

assert seed_choice is not None, (
    "Could not find a seed file with entry quotes, selected strike, date, and tenor. "
    "Inspect seed_inventory above."
)

seed_raw = pd.read_parquet(seed_choice).copy()

print("Using seed source:")
print(seed_choice)
print("Rows:", len(seed_raw))
print("Columns:", seed_raw.columns.tolist())

# Resolve source columns.
source_date_col = next(c for c in ["trade_dt", "trade_date", "trade_date_ts"] if c in seed_raw.columns)
source_tenor_col = next(c for c in ["entry_tenor", "tenor", "target_tenor"] if c in seed_raw.columns)

def first_existing_col(df, candidates, default_value=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default_value

def tenor_group_from_tenor(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

eod_seed = pd.DataFrame()

eod_seed["trade_dt"] = parse_project_date(seed_raw[source_date_col]).dt.normalize()
eod_seed["quote_timestamp"] = first_existing_col(seed_raw, ["quote_timestamp", "quote_time", "market_close_time"])
eod_seed["entry_tenor"] = seed_raw[source_tenor_col].astype(int)

eod_seed["expiry"] = parse_project_date(
    first_existing_col(seed_raw, ["selected_expiry_date", "target_expiry_date", "expiry"])
).dt.normalize()

eod_seed["actual_dte"] = first_existing_col(seed_raw, ["actual_dte"])
eod_seed["underlying_price"] = first_existing_col(seed_raw, ["entry_spx_close", "spx_close", "underlying_price"])
eod_seed["atm_strike"] = first_existing_col(seed_raw, ["short_strike_selected", "raw_short_strike", "atm_strike"])

eod_seed["atm_put_bid"] = first_existing_col(seed_raw, ["entry_bid", "atm_put_bid"])
eod_seed["atm_put_ask"] = first_existing_col(seed_raw, ["entry_ask", "atm_put_ask"])
eod_seed["atm_put_mid"] = first_existing_col(seed_raw, ["entry_mid", "atm_put_mid"])

eod_seed["atm_put_iv"] = first_existing_col(seed_raw, ["atm_put_iv", "vix_style_vol"])
eod_seed["atm_put_delta"] = first_existing_col(seed_raw, ["atm_put_delta", "put_delta"])

eod_seed["tenor_group"] = eod_seed["entry_tenor"].map(tenor_group_from_tenor)

# Keep useful research/live feature columns when present.
passthrough_cols = [
    "vix_style_vol",
    "implied_variance",
    "spx_rsi_14",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
    "pred_vol_pct_hybrid_back_har_only",
    "expiry_underlying_price",
    "expiry_spx_close",
    "expiry_intrinsic_value",
    "expired_otm",
    "entry_credit_points",
    "short_option_pnl_points",
    "normalized_pnl_dollars",
    "test_pnl",
    "win",
    "test_win",
    "pricing_status",
    "mapping_status",
    "selected_chain_file",
]

for c in passthrough_cols:
    if c in seed_raw.columns:
        eod_seed[c] = seed_raw[c]

# Keep only target tenors.
eod_seed = eod_seed[eod_seed["entry_tenor"].isin(TARGET_TENORS)].copy()

# Deduplicate one row per trade_dt / tenor.
eod_seed = (
    eod_seed
    .sort_values(["trade_dt", "entry_tenor"])
    .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
    .reset_index(drop=True)
)

# Basic validation.
expected_rows = eod_seed["trade_dt"].nunique() * len(TARGET_TENORS)
actual_rows = len(eod_seed)

print("EOD seed panel:")
print("Rows:", actual_rows)
print("Unique dates:", eod_seed["trade_dt"].nunique())
print("Date range:", eod_seed["trade_dt"].min(), "to", eod_seed["trade_dt"].max())
print("Unique tenors:", sorted(eod_seed["entry_tenor"].unique().tolist()))
print("Expected full grid rows:", expected_rows)
print("Missing grid rows:", expected_rows - actual_rows)

display(eod_seed.tail(20))

# Save as the new production EOD panel.
eod_seed.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

print("Saved seeded EOD panel:")
print(NAKED_ATM_EOD_PANEL)

Using seed source:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
Rows: 18099
Columns: ['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_ot

,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file
18079,2026-06-23,NaN,30,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.586777,NaN,back_27_33d,19.586777,0.038364,46.752076,46.752076,0.327861,-0.509629,-1.213937,-0.861783,0.313995,-0.596199,-1.266877,-1.266877,0.327861,-0.485334,-1.203989,-1.203989,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18080,2026-06-23,NaN,33,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.729159,NaN,back_27_33d,19.729159,0.038924,46.752076,46.752076,0.431584,-0.343885,-1.067693,-0.705789,0.377271,-0.512792,-1.208011,-1.208011,0.431584,-0.318612,-1.059780,-1.059780,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18081,2026-06-24,NaN,9,2026-07-02,8,7358.22,7355.0,69.8,70.9,70.35,19.326567,NaN,front_9_15d,19.326567,0.037352,46.342083,46.342083,0.567211,0.243603,-0.274788,-0.015593,0.669279,0.150559,-0.410467,-0.410467,0.567211,0.259376,-0.271507,-0.271507,NaN,NaN,None,69.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260702_160000.pkl
18082,2026-06-24,NaN,12,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.118185,NaN,front_9_15d,19.118185,0.036550,46.342083,46.342083,0.418108,-0.183775,-0.648595,-0.416185,0.739274,0.251112,-0.375552,-0.375552,0.418108,-0.162785,-0.644314,-0.644314,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18083,2026-06-24,NaN,15,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,18.992058,NaN,front_9_15d,18.992058,0.036070,46.342083,46.342083,0.068247,-0.766437,-1.282447,-1.024442,0.550836,-0.037813,-0.696963,-0.696963,0.068247,-0.740886,-1.271686,-1.271686,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18084,2026-06-24,NaN,18,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.091524,NaN,middle_18_24d,19.091524,0.036449,46.342083,46.342083,0.249885,-0.490428,-1.065457,-0.777942,0.621234,0.047563,-0.638722,-0.638722,0.249885,-0.466356,-1.057138,-1.057138,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18085,2026-06-24,NaN,21,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.240291,NaN,middle_18_24d,19.240291,0.037019,46.342083,46.342083,-0.002282,-1.133748,-1.627984,-1.380866,0.395771,-0.149625,-0.840562,-0.840562,-0.002282,-1.101711,-1.612012,-1.612012,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18086,2026-06-24,NaN,24,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.348982,NaN,middle_18_24d,19.348982,0.037438,46.342083,46.342083,0.113298,-0.899157,-1.475705,-1.187431,0.407037,-0.210791,-0.902639,-0.902639,0.113298,-0.873174,-1.462744,-1.462744,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18087,2026-06-24,NaN,27,2026-07-24,30,7358.22,7350.0,119.4,120.6,120.00,19.430528,NaN,back_27_33d,19.430528,0.037755,46.342083,46

Saved seeded EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet


In [20]:
# ============================================================
# Cell 7: Seed naked ATM EOD panel from historical all-tenor source
# ============================================================

seed_choice = None

preferred_files = [
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet",
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet",
]

for path in preferred_files:
    if not path.exists():
        continue

    temp = pd.read_parquet(path).copy()

    has_quotes = all(c in temp.columns for c in ["entry_bid", "entry_ask", "entry_mid"])
    has_strike = "short_strike_selected" in temp.columns
    has_tenor = any(c in temp.columns for c in ["entry_tenor", "tenor", "target_tenor"])
    has_date = any(c in temp.columns for c in ["trade_dt", "trade_date", "trade_date_ts"])

    if has_quotes and has_strike and has_tenor and has_date:
        seed_choice = path
        break

assert seed_choice is not None, (
    "Could not find a seed file with entry quotes, selected strike, date, and tenor. "
    "Inspect seed_inventory above."
)

seed_raw = pd.read_parquet(seed_choice).copy()

print("Using seed source:")
print(seed_choice)
print("Rows:", len(seed_raw))
print("Columns:", seed_raw.columns.tolist())

source_date_col = next(c for c in ["trade_dt", "trade_date", "trade_date_ts"] if c in seed_raw.columns)
source_tenor_col = next(c for c in ["entry_tenor", "tenor", "target_tenor"] if c in seed_raw.columns)

def first_existing_col(df, candidates, default_value=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default_value

def tenor_group_from_tenor(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

eod_seed = pd.DataFrame()

eod_seed["trade_dt"] = parse_project_date(seed_raw[source_date_col]).dt.normalize()
eod_seed["quote_timestamp"] = first_existing_col(seed_raw, ["quote_timestamp", "quote_time", "market_close_time"])
eod_seed["entry_tenor"] = seed_raw[source_tenor_col].astype(int)

eod_seed["expiry"] = parse_project_date(
    first_existing_col(seed_raw, ["selected_expiry_date", "target_expiry_date", "expiry"])
).dt.normalize()

eod_seed["actual_dte"] = first_existing_col(seed_raw, ["actual_dte"])
eod_seed["underlying_price"] = first_existing_col(seed_raw, ["entry_spx_close", "spx_close", "underlying_price"])
eod_seed["atm_strike"] = first_existing_col(seed_raw, ["short_strike_selected", "raw_short_strike", "atm_strike"])

eod_seed["atm_put_bid"] = first_existing_col(seed_raw, ["entry_bid", "atm_put_bid"])
eod_seed["atm_put_ask"] = first_existing_col(seed_raw, ["entry_ask", "atm_put_ask"])
eod_seed["atm_put_mid"] = first_existing_col(seed_raw, ["entry_mid", "atm_put_mid"])

eod_seed["atm_put_iv"] = first_existing_col(seed_raw, ["atm_put_iv", "vix_style_vol"])
eod_seed["atm_put_delta"] = first_existing_col(seed_raw, ["atm_put_delta", "put_delta"])

eod_seed["tenor_group"] = eod_seed["entry_tenor"].map(tenor_group_from_tenor)

passthrough_cols = [
    "vix_style_vol",
    "implied_variance",
    "spx_rsi_14",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
    "pred_vol_pct_hybrid_back_har_only",
    "expiry_underlying_price",
    "expiry_spx_close",
    "expiry_intrinsic_value",
    "expired_otm",
    "entry_credit_points",
    "short_option_pnl_points",
    "normalized_pnl_dollars",
    "test_pnl",
    "win",
    "test_win",
    "pricing_status",
    "mapping_status",
    "selected_chain_file",
]

for c in passthrough_cols:
    if c in seed_raw.columns:
        eod_seed[c] = seed_raw[c]

eod_seed = eod_seed[eod_seed["entry_tenor"].isin(TARGET_TENORS)].copy()

eod_seed = (
    eod_seed
    .sort_values(["trade_dt", "entry_tenor"])
    .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
    .reset_index(drop=True)
)

expected_rows = eod_seed["trade_dt"].nunique() * len(TARGET_TENORS)
actual_rows = len(eod_seed)

print("EOD seed panel:")
print("Rows:", actual_rows)
print("Unique dates:", eod_seed["trade_dt"].nunique())
print("Date range:", eod_seed["trade_dt"].min(), "to", eod_seed["trade_dt"].max())
print("Unique tenors:", sorted(eod_seed["entry_tenor"].unique().tolist()))
print("Expected full grid rows:", expected_rows)
print("Missing grid rows:", expected_rows - actual_rows)

display(eod_seed.tail(20))

eod_seed.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

print("Saved seeded EOD panel:")
print(NAKED_ATM_EOD_PANEL)

Using seed source:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
Rows: 18099
Columns: ['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_ot

,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file
18079,2026-06-23,NaN,30,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.586777,NaN,back_27_33d,19.586777,0.038364,46.752076,46.752076,0.327861,-0.509629,-1.213937,-0.861783,0.313995,-0.596199,-1.266877,-1.266877,0.327861,-0.485334,-1.203989,-1.203989,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18080,2026-06-23,NaN,33,2026-07-24,31,7365.46,7360.0,123.2,125.1,124.15,19.729159,NaN,back_27_33d,19.729159,0.038924,46.752076,46.752076,0.431584,-0.343885,-1.067693,-0.705789,0.377271,-0.512792,-1.208011,-1.208011,0.431584,-0.318612,-1.059780,-1.059780,NaN,NaN,None,123.2,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260623_20260724_160000.pkl
18081,2026-06-24,NaN,9,2026-07-02,8,7358.22,7355.0,69.8,70.9,70.35,19.326567,NaN,front_9_15d,19.326567,0.037352,46.342083,46.342083,0.567211,0.243603,-0.274788,-0.015593,0.669279,0.150559,-0.410467,-0.410467,0.567211,0.259376,-0.271507,-0.271507,NaN,NaN,None,69.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260702_160000.pkl
18082,2026-06-24,NaN,12,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.118185,NaN,front_9_15d,19.118185,0.036550,46.342083,46.342083,0.418108,-0.183775,-0.648595,-0.416185,0.739274,0.251112,-0.375552,-0.375552,0.418108,-0.162785,-0.644314,-0.644314,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18083,2026-06-24,NaN,15,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,18.992058,NaN,front_9_15d,18.992058,0.036070,46.342083,46.342083,0.068247,-0.766437,-1.282447,-1.024442,0.550836,-0.037813,-0.696963,-0.696963,0.068247,-0.740886,-1.271686,-1.271686,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18084,2026-06-24,NaN,18,2026-07-10,16,7358.22,7355.0,91.5,92.8,92.15,19.091524,NaN,middle_18_24d,19.091524,0.036449,46.342083,46.342083,0.249885,-0.490428,-1.065457,-0.777942,0.621234,0.047563,-0.638722,-0.638722,0.249885,-0.466356,-1.057138,-1.057138,NaN,NaN,None,91.5,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260624_20260710_160000.pkl
18085,2026-06-24,NaN,21,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.240291,NaN,middle_18_24d,19.240291,0.037019,46.342083,46.342083,-0.002282,-1.133748,-1.627984,-1.380866,0.395771,-0.149625,-0.840562,-0.840562,-0.002282,-1.101711,-1.612012,-1.612012,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18086,2026-06-24,NaN,24,2026-07-17,23,7358.22,7355.0,106.3,108.0,107.15,19.348982,NaN,middle_18_24d,19.348982,0.037438,46.342083,46.342083,0.113298,-0.899157,-1.475705,-1.187431,0.407037,-0.210791,-0.902639,-0.902639,0.113298,-0.873174,-1.462744,-1.462744,NaN,NaN,None,106.3,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260624_20260717_160000.pkl
18087,2026-06-24,NaN,27,2026-07-24,30,7358.22,7350.0,119.4,120.6,120.00,19.430528,NaN,back_27_33d,19.430528,0.037755,46.342083,46

Saved seeded EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet


In [21]:
# ============================================================
# Cell 8: Detect true missing EOD dates after seeding
# ============================================================

eod_panel = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()
eod_panel["trade_dt"] = pd.to_datetime(eod_panel["trade_dt"]).dt.normalize()

existing_dates = set(eod_panel["trade_dt"].dropna().unique())

BACKFILL_START_DATE = eod_panel["trade_dt"].min()
BACKFILL_END_DATE = trading_calendar.max()

target_dates = trading_calendar[
    (trading_calendar >= BACKFILL_START_DATE)
    & (trading_calendar <= BACKFILL_END_DATE)
]

missing_dates = [
    pd.Timestamp(d)
    for d in target_dates
    if pd.Timestamp(d) not in existing_dates
]

missing_dates_df = pd.DataFrame({"missing_trade_dt": missing_dates})

print("Seeded EOD panel rows:", len(eod_panel))
print("Seeded EOD panel date range:", eod_panel["trade_dt"].min(), "to", eod_panel["trade_dt"].max())
print("Trading calendar end:", trading_calendar.max())
print("Target trading days:", len(target_dates))
print("Missing dates:", len(missing_dates_df))

display(missing_dates_df.head(20))
display(missing_dates_df.tail(20))

tenor_count_by_date = (
    eod_panel
    .groupby("trade_dt")["entry_tenor"]
    .nunique()
    .reset_index(name="tenor_count")
)

incomplete_dates_df = tenor_count_by_date[
    tenor_count_by_date["tenor_count"] < len(TARGET_TENORS)
].copy()

print("Incomplete dates:", len(incomplete_dates_df))
display(incomplete_dates_df.tail(30))

Seeded EOD panel rows: 18099
Seeded EOD panel date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
Trading calendar end: 2026-06-25 00:00:00
Target trading days: 2011
Missing dates: 0


,missing_trade_dt


,missing_trade_dt


Incomplete dates: 0


,trade_dt,tenor_count


In [23]:
# ============================================================
# Cell 9: Validate seeded EOD panel and freshness
# ============================================================

eod_panel = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()

eod_panel["trade_dt"] = pd.to_datetime(eod_panel["trade_dt"]).dt.normalize()
eod_panel["entry_tenor"] = eod_panel["entry_tenor"].astype(int)

required_live_cols = [
    "trade_dt",
    "entry_tenor",
    "expiry",
    "actual_dte",
    "underlying_price",
    "atm_strike",
    "atm_put_bid",
    "atm_put_ask",
    "atm_put_mid",
    "atm_put_iv",
    "tenor_group",
    "vix_style_vol",
    "implied_variance",
    "entry_rsi_14",
    "old_vrp_signal",
    "old_vrp_3m_z",
    "old_vrp_1y_z",
    "old_vrp_blended_z",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
]

missing_required_cols = [
    c for c in required_live_cols
    if c not in eod_panel.columns
]

print("Missing required live columns:", missing_required_cols)

coverage = (
    eod_panel
    .groupby("trade_dt")
    .agg(
        tenor_count=("entry_tenor", "nunique"),
        min_tenor=("entry_tenor", "min"),
        max_tenor=("entry_tenor", "max"),
        missing_mid=("atm_put_mid", lambda x: x.isna().sum()),
        missing_iv=("atm_put_iv", lambda x: x.isna().sum()),
        missing_signal=("simple_carry_vrp_signal", lambda x: x.isna().sum()),
    )
    .reset_index()
)

bad_coverage = coverage[
    (coverage["tenor_count"] != len(TARGET_TENORS))
    | (coverage["missing_mid"] > 0)
    | (coverage["missing_iv"] > 0)
    | (coverage["missing_signal"] > 0)
].copy()

print("EOD panel rows:", len(eod_panel))
print("Date range:", eod_panel["trade_dt"].min(), "to", eod_panel["trade_dt"].max())
print("Unique dates:", eod_panel["trade_dt"].nunique())
print("Unique tenors:", sorted(eod_panel["entry_tenor"].unique().tolist()))
print("Bad coverage dates:", len(bad_coverage))

display(coverage.tail(10))
display(bad_coverage.tail(20))

Missing required live columns: []
EOD panel rows: 18099
Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
Unique dates: 2011
Unique tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Bad coverage dates: 9


,trade_dt,tenor_count,min_tenor,max_tenor,missing_mid,missing_iv,missing_signal
2001,2026-06-11,9,9,33,0,0,0
2002,2026-06-12,9,9,33,0,0,0
2003,2026-06-15,9,9,33,0,0,0
2004,2026-06-16,9,9,33,0,0,0
2005,2026-06-17,9,9,33,0,0,0
2006,2026-06-18,9,9,33,0,0,0
2007,2026-06-22,9,9,33,0,0,0
2008,2026-06-23,9,9,33,0,0,0
2009,2026-06-24,9,9,33,0,0,0
2010,2026-06-25,9,9,33,0,0,0


,trade_dt,tenor_count,min_tenor,max_tenor,missing_mid,missing_iv,missing_signal
0,2018-06-25,9,9,33,0,0,9
1,2018-06-26,9,9,33,0,0,9
2,2018-06-27,9,9,33,0,0,9
3,2018-06-28,9,9,33,0,0,9
4,2018-06-29,9,9,33,0,0,9
5,2018-07-02,9,9,33,0,0,9
6,2018-07-03,9,9,33,0,0,9
7,2018-07-05,9,9,33,0,0,9
8,2018-07-06,9,9,33,0,0,9


In [24]:
# ============================================================
# Cell 10: Identify EOD update dates after latest panel date
# ============================================================

# For production, set this manually when you know the latest date ThetaData has EOD data for.
# Example: UPDATE_THROUGH_DATE = "2026-07-01"
UPDATE_THROUGH_DATE = pd.Timestamp.today().normalize()

latest_panel_date = eod_panel["trade_dt"].max()

# Business-day approximation. Actual ThetaData fetch will decide what exists.
candidate_update_dates = pd.bdate_range(
    latest_panel_date + pd.offsets.BDay(1),
    UPDATE_THROUGH_DATE,
)

candidate_update_dates_df = pd.DataFrame({
    "candidate_update_dt": candidate_update_dates,
})

print("Latest EOD panel date:", latest_panel_date.date())
print("Update through date:", pd.Timestamp(UPDATE_THROUGH_DATE).date())
print("Candidate update dates:", len(candidate_update_dates_df))

display(candidate_update_dates_df)

Latest EOD panel date: 2026-06-25
Update through date: 2026-07-01
Candidate update dates: 4


,candidate_update_dt
0,2026-06-26
1,2026-06-29
2,2026-06-30
3,2026-07-01


In [25]:
# ============================================================
# Cell 11: Inventory existing ThetaData chain cache
# ============================================================

THETADATA_CHAIN_DIR = RAW_DATA_DIR / "thetadata_chains"

assert THETADATA_CHAIN_DIR.exists(), THETADATA_CHAIN_DIR

chain_files = sorted(THETADATA_CHAIN_DIR.glob("*.pkl"))

chain_rows = []

for path in chain_files:
    # Expected examples:
    # SPXW_20260625_20260724_160000.pkl
    # SPX_20260625_20260717_160000.pkl
    parts = path.stem.split("_")

    root = parts[0] if len(parts) > 0 else None
    trade_date_raw = parts[1] if len(parts) > 1 else None
    expiry_raw = parts[2] if len(parts) > 2 else None
    quote_time_raw = parts[3] if len(parts) > 3 else None

    trade_dt = pd.to_datetime(trade_date_raw, format="%Y%m%d", errors="coerce")
    expiry_dt = pd.to_datetime(expiry_raw, format="%Y%m%d", errors="coerce")

    chain_rows.append({
        "file": path.name,
        "path": str(path),
        "root": root,
        "trade_dt": trade_dt,
        "expiry": expiry_dt,
        "quote_time_raw": quote_time_raw,
        "size_mb": path.stat().st_size / 1_000_000,
    })

chain_inventory = pd.DataFrame(chain_rows)

print("Chain files:", len(chain_inventory))
print("Date range:", chain_inventory["trade_dt"].min(), "to", chain_inventory["trade_dt"].max())
print("Expiry range:", chain_inventory["expiry"].min(), "to", chain_inventory["expiry"].max())

display(
    chain_inventory
    .sort_values(["trade_dt", "expiry", "root"])
    .tail(30)
)

Chain files: 10901
Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
Expiry range: 2018-06-29 00:00:00 to 2026-07-31 00:00:00


,file,path,root,trade_dt,expiry,quote_time_raw,size_mb
10877,SPXW_20260617_20260702_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260617_20260702_160000.pkl,SPXW,2026-06-17,2026-07-02,160000,0.067109
10878,SPXW_20260617_20260710_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260617_20260710_160000.pkl,SPXW,2026-06-17,2026-07-10,160000,0.057638
2512,SPX_20260617_20260717_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260617_20260717_160000.pkl,SPX,2026-06-17,2026-07-17,160000,0.147599
10879,SPXW_20260617_20260724_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260617_20260724_160000.pkl,SPXW,2026-06-17,2026-07-24,160000,0.043943
10880,SPXW_20260618_20260626_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260618_20260626_160000.pkl,SPXW,2026-06-18,2026-06-26,160000,0.077567
10881,SPXW_20260618_20260702_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260618_20260702_160000.pkl,SPXW,2026-06-18,2026-07-02,160000,0.067358
10882,SPXW_20260618_20260710_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260618_20260710_160000.pkl,SPXW,2026-06-18,2026-07-10,160000,0.057887
2513,SPX_20260618_20260717_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260618_20260717_160000.pkl,SPX,2026-06-18,2026-07-17,160000,0.147599
10883,SPXW_20260618_20260724_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260618_20260724_160000.pkl,SPXW,2026-06-18,2026-07-24,160000,0.044441
10884,SPXW_20260622_20260626_160000.pkl,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260622_20260626_160000.pkl,SPXW,2026-06-22,2026-06-26,160000,0.077567


In [26]:
# ============================================================
# Cell 12: Inspect one recent ThetaData chain file structure
# ============================================================

import pickle

# Use the most recent cached chain file.
sample_chain_path = (
    chain_inventory
    .sort_values(["trade_dt", "expiry"])
    .iloc[-1]["path"]
)

print("Sample chain file:")
print(sample_chain_path)

with open(sample_chain_path, "rb") as f:
    sample_chain = pickle.load(f)

print("Loaded object type:", type(sample_chain))

if isinstance(sample_chain, pd.DataFrame):
    sample_chain_df = sample_chain.copy()
elif isinstance(sample_chain, dict):
    print("Dict keys:", sample_chain.keys())

    # Try to find first DataFrame inside dict.
    df_keys = [k for k, v in sample_chain.items() if isinstance(v, pd.DataFrame)]
    print("DataFrame keys:", df_keys)

    if df_keys:
        sample_chain_df = sample_chain[df_keys[0]].copy()
    else:
        sample_chain_df = pd.DataFrame(sample_chain)
else:
    sample_chain_df = pd.DataFrame(sample_chain)

print("Sample chain rows:", len(sample_chain_df))
print("Sample chain columns:")
print(sample_chain_df.columns.tolist())

display(sample_chain_df.head())
display(sample_chain_df.tail())

Sample chain file:
C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260731_160000.pkl
Loaded object type: <class 'pandas.core.frame.DataFrame'>
Sample chain rows: 956
Sample chain columns:
['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'bid_size', 'ask_size', 'bid_exchange', 'ask_exchange', 'bid_condition', 'ask_condition', 'timestamp']


,root,expiration,strike,right,bid,ask,mid,bid_size,ask_size,bid_exchange,ask_exchange,bid_condition,ask_condition,timestamp
0,SPXW,20260731,7635.0,C,41.6,42.6,42.10,6,22,5,5,50,50,2026-06-25T16:00:00.000
1,SPXW,20260731,6910.0,P,39.5,40.3,39.90,2,6,5,5,50,50,2026-06-25T16:00:00.000
2,SPXW,20260731,6910.0,C,509.5,529.5,519.50,1,1,5,5,50,50,2026-06-25T16:00:00.000
3,SPXW,20260731,7465.0,C,108.2,109.5,108.85,5,1,5,5,50,50,2026-06-25T16:00:00.000
4,SPXW,20260731,7080.0,C,371.6,382.3,376.95,19,1,5,5,50,50,2026-06-25T16:00:00.000


,root,expiration,strike,right,bid,ask,mid,bid_size,ask_size,bid_exchange,ask_exchange,bid_condition,ask_condition,timestamp
951,SPXW,20260731,6350.0,C,1038.2,1056.8,1047.50,1,1,5,5,50,50,2026-06-25T16:00:00.000
952,SPXW,20260731,7460.0,C,110.6,112.1,111.35,5,1,5,5,50,50,2026-06-25T16:00:00.000
953,SPXW,20260731,6905.0,P,39.0,39.8,39.40,18,6,5,5,50,50,2026-06-25T16:00:00.000
954,SPXW,20260731,7460.0,P,176.4,181.0,178.70,7,26,5,5,50,50,2026-06-25T16:00:00.000
955,SPXW,20260731,6350.0,P,11.8,12.3,12.05,10,78,5,5,50,50,2026-06-25T16:00:00.000


In [33]:
# ============================================================
# Cell 13: Normalize raw ThetaData chain format
# ============================================================

def normalize_chain_df(raw_chain):
    """
    Convert a raw cached ThetaData chain object into a standard option quote DataFrame.

    Output target columns:
        root
        expiration
        strike
        right
        bid
        ask
        mid
        iv
        delta
        timestamp
    """

    if isinstance(raw_chain, pd.DataFrame):
        df = raw_chain.copy()
    elif isinstance(raw_chain, dict):
        df_keys = [k for k, v in raw_chain.items() if isinstance(v, pd.DataFrame)]
        if df_keys:
            df = raw_chain[df_keys[0]].copy()
        else:
            df = pd.DataFrame(raw_chain)
    else:
        df = pd.DataFrame(raw_chain)

    lower_map = {c.lower(): c for c in df.columns}

    def find_col(candidates):
        for c in candidates:
            if c.lower() in lower_map:
                return lower_map[c.lower()]
        return None

    col_root = find_col(["root", "symbol", "underlying", "option_root"])
    col_exp = find_col(["expiration", "expiry", "exp", "expiration_date"])
    col_strike = find_col(["strike", "strike_price"])
    col_right = find_col(["right", "put_call", "cp", "option_type", "type"])
    col_bid = find_col(["bid", "bid_price"])
    col_ask = find_col(["ask", "ask_price"])
    col_mid = find_col(["mid", "mark", "price"])
    col_iv = find_col(["iv", "implied_vol", "implied_volatility"])
    col_delta = find_col(["delta"])
    col_timestamp = find_col(["timestamp", "quote_timestamp", "time", "datetime"])

    out = pd.DataFrame(index=df.index)

    out["root"] = df[col_root] if col_root is not None else np.nan
    out["expiration"] = parse_project_date(df[col_exp]).dt.normalize() if col_exp is not None else pd.NaT
    out["strike"] = pd.to_numeric(df[col_strike], errors="coerce") if col_strike is not None else np.nan
    out["right"] = df[col_right].astype(str).str.lower() if col_right is not None else np.nan
    out["bid"] = pd.to_numeric(df[col_bid], errors="coerce") if col_bid is not None else np.nan
    out["ask"] = pd.to_numeric(df[col_ask], errors="coerce") if col_ask is not None else np.nan

    if col_mid is not None:
        out["mid"] = pd.to_numeric(df[col_mid], errors="coerce")
    else:
        out["mid"] = (out["bid"] + out["ask"]) / 2.0

    out["iv"] = pd.to_numeric(df[col_iv], errors="coerce") if col_iv is not None else np.nan
    out["delta"] = pd.to_numeric(df[col_delta], errors="coerce") if col_delta is not None else np.nan
    out["timestamp"] = df[col_timestamp] if col_timestamp is not None else np.nan

    out["right"] = out["right"].replace({
        "p": "put",
        "put": "put",
        "c": "call",
        "call": "call",
    })

    return out


normalized_sample_chain = normalize_chain_df(sample_chain)

print("Normalized sample chain:")
print(normalized_sample_chain.columns.tolist())
print(normalized_sample_chain.shape)

display(normalized_sample_chain.head())
display(normalized_sample_chain.tail())
display(normalized_sample_chain["right"].value_counts(dropna=False))

Normalized sample chain:
['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'iv', 'delta', 'timestamp']
(956, 10)


,root,expiration,strike,right,bid,ask,mid,iv,delta,timestamp
0,SPXW,2026-07-31,7635.0,call,41.6,42.6,42.10,NaN,NaN,2026-06-25T16:00:00.000
1,SPXW,2026-07-31,6910.0,put,39.5,40.3,39.90,NaN,NaN,2026-06-25T16:00:00.000
2,SPXW,2026-07-31,6910.0,call,509.5,529.5,519.50,NaN,NaN,2026-06-25T16:00:00.000
3,SPXW,2026-07-31,7465.0,call,108.2,109.5,108.85,NaN,NaN,2026-06-25T16:00:00.000
4,SPXW,2026-07-31,7080.0,call,371.6,382.3,376.95,NaN,NaN,2026-06-25T16:00:00.000


,root,expiration,strike,right,bid,ask,mid,iv,delta,timestamp
951,SPXW,2026-07-31,6350.0,call,1038.2,1056.8,1047.50,NaN,NaN,2026-06-25T16:00:00.000
952,SPXW,2026-07-31,7460.0,call,110.6,112.1,111.35,NaN,NaN,2026-06-25T16:00:00.000
953,SPXW,2026-07-31,6905.0,put,39.0,39.8,39.40,NaN,NaN,2026-06-25T16:00:00.000
954,SPXW,2026-07-31,7460.0,put,176.4,181.0,178.70,NaN,NaN,2026-06-25T16:00:00.000
955,SPXW,2026-07-31,6350.0,put,11.8,12.3,12.05,NaN,NaN,2026-06-25T16:00:00.000


right
call    478
put     478
Name: count, dtype: int64

In [34]:
# ============================================================
# Cell 14: Select ATM put from normalized chain
# Historical-compatible rule: highest put strike <= underlying
# ============================================================

def select_atm_put_from_chain(chain_df, underlying_price):
    """
    Select ATM put from a normalized chain.

    Historical-compatible rule:
        choose highest put strike <= underlying_price.

    This selects the closest non-ITM put. If no strike is below spot,
    fallback to nearest strike.
    """

    puts = chain_df[
        chain_df["right"].eq("put")
    ].copy()

    puts = puts[
        puts["strike"].notna()
        & puts["bid"].notna()
        & puts["ask"].notna()
        & puts["mid"].notna()
    ].copy()

    if puts.empty:
        return None

    puts["strike_distance"] = (puts["strike"] - underlying_price).abs()
    puts["put_moneyness"] = puts["strike"] / underlying_price
    puts["put_pct_otm"] = 1.0 - puts["put_moneyness"]

    non_itm_puts = puts[puts["strike"] <= underlying_price].copy()

    if not non_itm_puts.empty:
        selected = (
            non_itm_puts
            .sort_values(["strike"], ascending=False)
            .iloc[0]
        )
    else:
        selected = (
            puts
            .sort_values(["strike_distance", "strike"])
            .iloc[0]
        )

    return selected


sample_underlying = 7357.49

sample_atm_put = select_atm_put_from_chain(
    normalized_sample_chain,
    sample_underlying,
)

print("Sample ATM put:")
display(sample_atm_put.to_frame().T)

Sample ATM put:


,root,expiration,strike,right,bid,ask,mid,iv,delta,timestamp,strike_distance,put_moneyness,put_pct_otm
692,SPXW,2026-07-31 00:00:00,7355.0,put,132.0,133.2,132.6,NaN,NaN,2026-06-25T16:00:00.000,2.49,0.999662,0.000338


In [29]:
# ============================================================
# Cell 15: Read cached ThetaData chain by trade date and expiry
# ============================================================

def find_cached_chain_file(trade_dt, expiry, preferred_roots=("SPXW", "SPX"), quote_time="160000"):
    """
    Find cached ThetaData chain file for trade_dt / expiry.

    Looks for:
        ROOT_YYYYMMDD_YYYYMMDD_160000.pkl

    Prefers SPXW, then SPX by default.
    """

    trade_dt = pd.Timestamp(trade_dt).strftime("%Y%m%d")
    expiry = pd.Timestamp(expiry).strftime("%Y%m%d")

    candidates = []

    for root in preferred_roots:
        path = THETADATA_CHAIN_DIR / f"{root}_{trade_dt}_{expiry}_{quote_time}.pkl"
        candidates.append(path)

        if path.exists():
            return path

    return None


def read_cached_chain_file(path):
    """
    Read cached pickle chain file and normalize.
    """

    import pickle

    with open(path, "rb") as f:
        raw_chain = pickle.load(f)

    normalized = normalize_chain_df(raw_chain)

    return normalized


# Test using sample file path.
sample_cached_path = Path(sample_chain_path)

sample_read_chain = read_cached_chain_file(sample_cached_path)

print("Read cached chain:")
print(sample_cached_path)
print(sample_read_chain.shape)
display(sample_read_chain.head())

Read cached chain:
C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260731_160000.pkl
(956, 9)


,root,expiration,strike,right,bid,ask,mid,iv,delta
0,SPXW,2026-07-31,7635.0,call,41.6,42.6,42.10,NaN,NaN
1,SPXW,2026-07-31,6910.0,put,39.5,40.3,39.90,NaN,NaN
2,SPXW,2026-07-31,6910.0,call,509.5,529.5,519.50,NaN,NaN
3,SPXW,2026-07-31,7465.0,call,108.2,109.5,108.85,NaN,NaN
4,SPXW,2026-07-31,7080.0,call,371.6,382.3,376.95,NaN,NaN


In [30]:
# ============================================================
# Cell 16: Build EOD ATM rows for one date from cached chains
# ============================================================

def build_eod_atm_rows_from_cache(trade_dt, base_panel=eod_panel):
    """
    Build one EOD ATM row per target tenor for a trade date.

    Uses base_panel for:
        selected expiry
        underlying price
        signal / feature columns

    Uses cached ThetaData chains for:
        ATM strike
        bid / ask / mid
        quote timestamp
    """

    trade_dt = pd.Timestamp(trade_dt).normalize()

    day_panel = base_panel[
        base_panel["trade_dt"].eq(trade_dt)
    ].copy()

    if day_panel.empty:
        raise ValueError(f"No base panel rows found for {trade_dt.date()}")

    rows = []

    for _, base_row in day_panel.sort_values("entry_tenor").iterrows():
        tenor = int(base_row["entry_tenor"])
        expiry = pd.Timestamp(base_row["expiry"]).normalize()
        underlying_price = float(base_row["underlying_price"])

        chain_path = find_cached_chain_file(
            trade_dt=trade_dt,
            expiry=expiry,
        )

        if chain_path is None:
            out = base_row.to_dict()
            out["cache_status"] = "missing_chain_file"
            out["selected_chain_file_live"] = None
            rows.append(out)
            continue

        chain = read_cached_chain_file(chain_path)

        selected_put = select_atm_put_from_chain(
            chain,
            underlying_price=underlying_price,
        )

        if selected_put is None:
            out = base_row.to_dict()
            out["cache_status"] = "no_valid_put"
            out["selected_chain_file_live"] = str(chain_path)
            rows.append(out)
            continue

        out = base_row.to_dict()

        out["cache_status"] = "mapped"
        out["selected_chain_file_live"] = str(chain_path)

        out["atm_strike"] = float(selected_put["strike"])
        out["atm_put_bid"] = float(selected_put["bid"])
        out["atm_put_ask"] = float(selected_put["ask"])
        out["atm_put_mid"] = float(selected_put["mid"])
        out["atm_put_iv"] = selected_put["iv"] if "iv" in selected_put.index else np.nan
        out["atm_put_delta"] = selected_put["delta"] if "delta" in selected_put.index else np.nan

        out["put_moneyness"] = float(selected_put["put_moneyness"])
        out["put_pct_otm"] = float(selected_put["put_pct_otm"])

        if "timestamp" in chain.columns:
            out["quote_timestamp"] = chain["timestamp"].dropna().iloc[0] if chain["timestamp"].notna().any() else np.nan

        rows.append(out)

    result = pd.DataFrame(rows)

    result["trade_dt"] = pd.to_datetime(result["trade_dt"]).dt.normalize()
    result["entry_tenor"] = result["entry_tenor"].astype(int)

    return result


# Test on latest seeded date.
test_trade_dt = eod_panel["trade_dt"].max()

test_eod_rows = build_eod_atm_rows_from_cache(test_trade_dt)

print("Test trade date:", test_trade_dt.date())
print("Rows:", len(test_eod_rows))
display(
    test_eod_rows[
        [
            "trade_dt",
            "entry_tenor",
            "expiry",
            "actual_dte",
            "underlying_price",
            "atm_strike",
            "atm_put_bid",
            "atm_put_ask",
            "atm_put_mid",
            "put_moneyness",
            "cache_status",
            "selected_chain_file_live",
        ]
    ]
)

Test trade date: 2026-06-25
Rows: 9


,trade_dt,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,put_moneyness,cache_status,selected_chain_file_live
0,2026-06-25,9,2026-07-02,7,7357.49,7355.0,59.9,61.5,60.70,0.999662,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl
1,2026-06-25,12,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,0.999662,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl
2,2026-06-25,15,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,0.999662,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl
3,2026-06-25,18,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,0.999662,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl
4,2026-06-25,21,2026-07-17,22,7357.49,7355.0,100.8,101.9,101.35,0.999662,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl
5,2026-06-25,24,2026-07-17,22,7357.49,7355.0,100.8,101.9,101.35,0.999662,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl
6,2026-06-25,27,2026-07-24,29,7357.49,7360.0,119.2,120.5,119.85,1.000341,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260724_160000.pkl
7,2026-06-25,30,2026-07-24,29,7357.49,7360.0,119.2,120.5,119.85,1.000341,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260724_160000.pkl
8,2026-06-25,33,2026-07-31,36,7357.49,7355.0,132.0,133.2,132.60,0.999662,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260731_160000.pkl


In [36]:
# ============================================================
# Cell 17: Validate cache rebuild versus seeded panel
# ============================================================

def compare_cache_rebuild_to_panel(trade_dt):
    trade_dt = pd.Timestamp(trade_dt).normalize()

    rebuilt = build_eod_atm_rows_from_cache(trade_dt).copy()

    original = eod_panel[
        eod_panel["trade_dt"].eq(trade_dt)
    ].copy()

    compare = original[
        [
            "trade_dt",
            "entry_tenor",
            "expiry",
            "underlying_price",
            "atm_strike",
            "atm_put_bid",
            "atm_put_ask",
            "atm_put_mid",
        ]
    ].merge(
        rebuilt[
            [
                "trade_dt",
                "entry_tenor",
                "atm_strike",
                "atm_put_bid",
                "atm_put_ask",
                "atm_put_mid",
                "cache_status",
            ]
        ],
        on=["trade_dt", "entry_tenor"],
        how="left",
        suffixes=("_original", "_rebuilt"),
    )

    for c in ["atm_strike", "atm_put_bid", "atm_put_ask", "atm_put_mid"]:
        compare[f"{c}_diff"] = compare[f"{c}_rebuilt"] - compare[f"{c}_original"]

    return compare


validation_dates = [
    eod_panel["trade_dt"].max(),
    eod_panel["trade_dt"].drop_duplicates().iloc[-2],
    eod_panel["trade_dt"].drop_duplicates().iloc[-10],
]

validation_results = []

for dt in validation_dates:
    temp = compare_cache_rebuild_to_panel(dt)
    validation_results.append(temp)

validation_df = pd.concat(validation_results, ignore_index=True)

display(validation_df)

print("Max absolute differences:")
diff_cols = [c for c in validation_df.columns if c.endswith("_diff")]
display(validation_df[diff_cols].abs().max().to_frame("max_abs_diff"))

,trade_dt,entry_tenor,expiry,underlying_price,atm_strike_original,atm_put_bid_original,atm_put_ask_original,atm_put_mid_original,atm_strike_rebuilt,atm_put_bid_rebuilt,atm_put_ask_rebuilt,atm_put_mid_rebuilt,cache_status,atm_strike_diff,atm_put_bid_diff,atm_put_ask_diff,atm_put_mid_diff
0,2026-06-25,9,2026-07-02,7357.49,7355.0,59.9,61.5,60.70,7355.0,59.9,61.5,60.70,mapped,0.0,0.0,0.0,0.0
1,2026-06-25,12,2026-07-10,7357.49,7355.0,84.8,86.2,85.50,7355.0,84.8,86.2,85.50,mapped,0.0,0.0,0.0,0.0
2,2026-06-25,15,2026-07-10,7357.49,7355.0,84.8,86.2,85.50,7355.0,84.8,86.2,85.50,mapped,0.0,0.0,0.0,0.0
3,2026-06-25,18,2026-07-10,7357.49,7355.0,84.8,86.2,85.50,7355.0,84.8,86.2,85.50,mapped,0.0,0.0,0.0,0.0
4,2026-06-25,21,2026-07-17,7357.49,7355.0,100.8,101.9,101.35,7355.0,100.8,101.9,101.35,mapped,0.0,0.0,0.0,0.0
5,2026-06-25,24,2026-07-17,7357.49,7355.0,100.8,101.9,101.35,7355.0,100.8,101.9,101.35,mapped,0.0,0.0,0.0,0.0
6,2026-06-25,27,2026-07-24,7357.49,7350.0,115.6,116.8,116.20,7350.0,115.6,116.8,116.20,mapped,0.0,0.0,0.0,0.0
7,2026-06-25,30,2026-07-24,7357.49,7350.0,115.6,116.8,116.20,7350.0,115.6,116.8,116.20,mapped,0.0,0.0,0.0,0.0
8,2026-06-25,33,2026-07-31,7357.49,7355.0,132.0,133.2,132.60,7355.0,132.0,133.2,132.60,mapped,0.0,0.0,0.0,0.0
9,2026-06-24,9,2026-07-02,7358.22,7355.0,69.8,70.9,70.35,7355.0,69.8,70.9,70.35,mapped,0.0,0.0,0.0,0.0


Max absolute differences:


,max_abs_diff
atm_strike_diff,0.0
atm_put_bid_diff,0.0
atm_put_ask_diff,0.0
atm_put_mid_diff,0.0


In [37]:
# ============================================================
# Cell 18: Append / refresh EOD panel dates from cached chains
# ============================================================

def refresh_eod_panel_from_cache(update_dates, overwrite_existing=True):
    """
    Refresh EOD panel rows for update_dates using cached ThetaData chain files.

    This assumes:
        1. Base feature rows already exist for those dates in eod_panel or another feature panel.
        2. Cached chain files exist under data/raw/thetadata_chains.

    For dates beyond the current feature panel, we will later need to build/update
    the feature panel first.
    """

    current_panel = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()
    current_panel["trade_dt"] = pd.to_datetime(current_panel["trade_dt"]).dt.normalize()

    new_rows = []

    for dt in update_dates:
        dt = pd.Timestamp(dt).normalize()

        try:
            day_rows = build_eod_atm_rows_from_cache(dt, base_panel=current_panel)
            new_rows.append(day_rows)
            print(f"{dt.date()}: refreshed {len(day_rows)} rows")
        except Exception as e:
            print(f"{dt.date()}: ERROR - {e}")

    if not new_rows:
        print("No rows refreshed.")
        return current_panel

    new_panel_rows = pd.concat(new_rows, ignore_index=True)
    new_panel_rows["trade_dt"] = pd.to_datetime(new_panel_rows["trade_dt"]).dt.normalize()

    if overwrite_existing:
        replace_dates = set(new_panel_rows["trade_dt"].unique())
        current_panel = current_panel[
            ~current_panel["trade_dt"].isin(replace_dates)
        ].copy()

    updated_panel = pd.concat(
        [current_panel, new_panel_rows],
        ignore_index=True,
    )

    updated_panel = (
        updated_panel
        .sort_values(["trade_dt", "entry_tenor"])
        .drop_duplicates(["trade_dt", "entry_tenor"], keep="last")
        .reset_index(drop=True)
    )

    updated_panel.to_parquet(NAKED_ATM_EOD_PANEL, index=False)

    print("Saved updated EOD panel:")
    print(NAKED_ATM_EOD_PANEL)
    print("Rows:", len(updated_panel))
    print("Date range:", updated_panel["trade_dt"].min(), "to", updated_panel["trade_dt"].max())

    return updated_panel


# Dry test: refresh latest existing date only.
REFRESH_TEST_DATES = [eod_panel["trade_dt"].max()]

refreshed_test_panel = refresh_eod_panel_from_cache(
    REFRESH_TEST_DATES,
    overwrite_existing=True,
)

2026-06-25: refreshed 9 rows
Saved updated EOD panel:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet
Rows: 18099
Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00


In [38]:
# ============================================================
# Cell 19: ThetaData cache file naming and downloader contract
# ============================================================

def make_chain_cache_path(root, trade_dt, expiry, quote_time="160000"):
    """
    Standard cache path:
        data/raw/thetadata_chains/ROOT_YYYYMMDD_YYYYMMDD_HHMMSS.pkl
    """

    trade_str = pd.Timestamp(trade_dt).strftime("%Y%m%d")
    expiry_str = pd.Timestamp(expiry).strftime("%Y%m%d")

    return THETADATA_CHAIN_DIR / f"{root}_{trade_str}_{expiry_str}_{quote_time}.pkl"


def expected_chain_cache_paths(trade_dt, expiries, roots=("SPXW", "SPX"), quote_time="160000"):
    """
    Return all expected root/date/expiry cache paths.
    """

    rows = []

    for expiry in sorted(set(pd.to_datetime(expiries).normalize())):
        for root in roots:
            rows.append({
                "root": root,
                "trade_dt": pd.Timestamp(trade_dt).normalize(),
                "expiry": pd.Timestamp(expiry).normalize(),
                "quote_time": quote_time,
                "path": make_chain_cache_path(root, trade_dt, expiry, quote_time),
            })

    return pd.DataFrame(rows)


# Example expected paths for latest date.
latest_dt = eod_panel["trade_dt"].max()
latest_expiries = eod_panel.loc[eod_panel["trade_dt"].eq(latest_dt), "expiry"].unique()

expected_latest_paths = expected_chain_cache_paths(latest_dt, latest_expiries)

expected_latest_paths["exists"] = expected_latest_paths["path"].apply(lambda p: p.exists())

display(expected_latest_paths)

,root,trade_dt,expiry,quote_time,path,exists
0,SPXW,2026-06-25,2026-07-02,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl,True
1,SPX,2026-06-25,2026-07-02,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260702_160000.pkl,False
2,SPXW,2026-06-25,2026-07-10,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,True
3,SPX,2026-06-25,2026-07-10,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260710_160000.pkl,False
4,SPXW,2026-06-25,2026-07-17,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260717_160000.pkl,False
5,SPX,2026-06-25,2026-07-17,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,True
6,SPXW,2026-06-25,2026-07-24,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260724_160000.pkl,True
7,SPX,2026-06-25,2026-07-24,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260724_160000.pkl,False
8,SPXW,2026-06-25,2026-07-31,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260731_160000.pkl,True
9,SPX,2026-06-25,2026-07-31,160000,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260731_160000.pkl,False


In [39]:
# ============================================================
# Cell 20: Check cache completeness for candidate update dates
# ============================================================

def get_required_expiries_for_date_from_panel(trade_dt, panel=eod_panel):
    """
    For dates already in the panel, use its mapped expiries.
    For future dates not yet in the panel, this will need an expiry-mapping step.
    """

    trade_dt = pd.Timestamp(trade_dt).normalize()

    day = panel[panel["trade_dt"].eq(trade_dt)].copy()

    if day.empty:
        return []

    return sorted(pd.to_datetime(day["expiry"]).dt.normalize().unique())


def check_cache_for_dates(update_dates, panel=eod_panel):
    rows = []

    for dt in update_dates:
        dt = pd.Timestamp(dt).normalize()
        expiries = get_required_expiries_for_date_from_panel(dt, panel=panel)

        if not expiries:
            rows.append({
                "trade_dt": dt,
                "expiry": pd.NaT,
                "root": None,
                "path": None,
                "exists": False,
                "status": "no_expiry_mapping_in_panel",
            })
            continue

        expected = expected_chain_cache_paths(dt, expiries)
        expected["exists"] = expected["path"].apply(lambda p: p.exists())

        for _, row in expected.iterrows():
            rows.append({
                "trade_dt": dt,
                "expiry": row["expiry"],
                "root": row["root"],
                "path": str(row["path"]),
                "exists": bool(row["exists"]),
                "status": "exists" if row["exists"] else "missing",
            })

    return pd.DataFrame(rows)


# Test against latest existing date.
cache_check_latest = check_cache_for_dates([latest_dt])

display(cache_check_latest)

,trade_dt,expiry,root,path,exists,status
0,2026-06-25,2026-07-02,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl,True,exists
1,2026-06-25,2026-07-02,SPX,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260702_160000.pkl,False,missing
2,2026-06-25,2026-07-10,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,True,exists
3,2026-06-25,2026-07-10,SPX,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260710_160000.pkl,False,missing
4,2026-06-25,2026-07-17,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260717_160000.pkl,False,missing
5,2026-06-25,2026-07-17,SPX,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,True,exists
6,2026-06-25,2026-07-24,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260724_160000.pkl,True,exists
7,2026-06-25,2026-07-24,SPX,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260724_160000.pkl,False,missing
8,2026-06-25,2026-07-31,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260731_160000.pkl,True,exists
9,2026-06-25,2026-07-31,SPX,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260731_160000.pkl,False,missing


In [40]:
# ============================================================
# Cell 21: ThetaData downloader stub
# Wire this function to your local ThetaData client
# ============================================================

def download_thetadata_chain_to_cache(root, trade_dt, expiry, quote_time="160000", overwrite=False):
    """
    Download one option chain snapshot from ThetaData and save to cache.

    Required output format:
        Pickled pandas DataFrame with columns like:
            root, expiration, strike, right, bid, ask, mid, timestamp

    Cache path:
        data/raw/thetadata_chains/ROOT_YYYYMMDD_YYYYMMDD_HHMMSS.pkl

    Return:
        path to saved cache file

    TODO:
        Replace the NotImplementedError block with your actual ThetaData call.
    """

    cache_path = make_chain_cache_path(
        root=root,
        trade_dt=trade_dt,
        expiry=expiry,
        quote_time=quote_time,
    )

    if cache_path.exists() and not overwrite:
        return cache_path

    raise NotImplementedError(
        "Wire this to your ThetaData client. "
        "It should save a pickled DataFrame to cache_path."
    )


def ensure_chain_cache_for_date(trade_dt, panel=eod_panel, roots=("SPXW", "SPX"), quote_time="160000", overwrite=False):
    """
    Ensure all required chain files exist for a trade date already present in the panel.

    For each required expiry, tries roots in order.
    """

    trade_dt = pd.Timestamp(trade_dt).normalize()
    expiries = get_required_expiries_for_date_from_panel(trade_dt, panel=panel)

    if not expiries:
        raise ValueError(f"No expiry mapping available for {trade_dt.date()}")

    rows = []

    for expiry in expiries:
        expiry = pd.Timestamp(expiry).normalize()

        root_success = False

        for root in roots:
            cache_path = make_chain_cache_path(root, trade_dt, expiry, quote_time)

            if cache_path.exists() and not overwrite:
                rows.append({
                    "trade_dt": trade_dt,
                    "expiry": expiry,
                    "root": root,
                    "cache_path": str(cache_path),
                    "status": "exists",
                })
                root_success = True
                break

            try:
                saved_path = download_thetadata_chain_to_cache(
                    root=root,
                    trade_dt=trade_dt,
                    expiry=expiry,
                    quote_time=quote_time,
                    overwrite=overwrite,
                )

                rows.append({
                    "trade_dt": trade_dt,
                    "expiry": expiry,
                    "root": root,
                    "cache_path": str(saved_path),
                    "status": "downloaded",
                })
                root_success = True
                break

            except NotImplementedError:
                rows.append({
                    "trade_dt": trade_dt,
                    "expiry": expiry,
                    "root": root,
                    "cache_path": str(cache_path),
                    "status": "downloader_not_implemented",
                })
                break

            except Exception as e:
                rows.append({
                    "trade_dt": trade_dt,
                    "expiry": expiry,
                    "root": root,
                    "cache_path": str(cache_path),
                    "status": f"error: {e}",
                })

        if not root_success:
            rows.append({
                "trade_dt": trade_dt,
                "expiry": expiry,
                "root": None,
                "cache_path": None,
                "status": "failed_all_roots",
            })

    return pd.DataFrame(rows)


# Test on latest existing date. Should mostly say "exists".
ensure_latest = ensure_chain_cache_for_date(latest_dt)

display(ensure_latest)

,trade_dt,expiry,root,cache_path,status
0,2026-06-25,2026-07-02,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl,exists
1,2026-06-25,2026-07-10,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,exists
2,2026-06-25,2026-07-17,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260717_160000.pkl,downloader_not_implemented
3,2026-06-25,2026-07-17,None,None,failed_all_roots
4,2026-06-25,2026-07-24,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260724_160000.pkl,exists
5,2026-06-25,2026-07-31,SPXW,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260731_160000.pkl,exists


In [41]:
# ============================================================
# Cell 22: Build intraday ATM snapshot from cache
# ============================================================

def build_intraday_snapshot_from_cache(
    snapshot_trade_dt,
    snapshot_quote_time,
    base_panel=eod_panel,
    output_path=NAKED_ATM_INTRADAY_SNAPSHOT,
):
    """
    Build intraday ATM snapshot for one date and quote time.

    Assumes chain files already exist in cache:
        ROOT_YYYYMMDD_EXPIRY_YYYYMMDD_HHMMSS.pkl

    For now, uses base_panel for:
        mapped expiries
        feature rows
        prior signal columns

    In live intraday mode, underlying_price should be updated from current spot.
    For MVP, this uses the panel underlying price unless later replaced.
    """

    snapshot_trade_dt = pd.Timestamp(snapshot_trade_dt).normalize()
    quote_time = str(snapshot_quote_time)

    day_panel = base_panel[
        base_panel["trade_dt"].eq(snapshot_trade_dt)
    ].copy()

    if day_panel.empty:
        raise ValueError(
            f"No base panel rows for {snapshot_trade_dt.date()}. "
            "Need intraday expiry/feature mapping for new live dates."
        )

    rows = []

    for _, base_row in day_panel.sort_values("entry_tenor").iterrows():
        tenor = int(base_row["entry_tenor"])
        expiry = pd.Timestamp(base_row["expiry"]).normalize()
        underlying_price = float(base_row["underlying_price"])

        chain_path = find_cached_chain_file(
            trade_dt=snapshot_trade_dt,
            expiry=expiry,
            quote_time=quote_time,
        )

        if chain_path is None:
            out = base_row.to_dict()
            out["snapshot_trade_dt"] = snapshot_trade_dt
            out["snapshot_quote_time"] = quote_time
            out["intraday_status"] = "missing_chain_file"
            out["selected_chain_file_intraday"] = None
            rows.append(out)
            continue

        chain = read_cached_chain_file(chain_path)
        selected_put = select_atm_put_from_chain(chain, underlying_price=underlying_price)

        out = base_row.to_dict()
        out["snapshot_trade_dt"] = snapshot_trade_dt
        out["snapshot_quote_time"] = quote_time
        out["selected_chain_file_intraday"] = str(chain_path)

        if selected_put is None:
            out["intraday_status"] = "no_valid_put"
        else:
            out["intraday_status"] = "mapped"
            out["atm_strike"] = float(selected_put["strike"])
            out["atm_put_bid"] = float(selected_put["bid"])
            out["atm_put_ask"] = float(selected_put["ask"])
            out["atm_put_mid"] = float(selected_put["mid"])
            out["atm_put_iv"] = selected_put["iv"] if "iv" in selected_put.index else np.nan
            out["atm_put_delta"] = selected_put["delta"] if "delta" in selected_put.index else np.nan
            out["put_moneyness"] = float(selected_put["put_moneyness"])
            out["put_pct_otm"] = float(selected_put["put_pct_otm"])

            if "timestamp" in selected_put.index:
                out["quote_timestamp"] = selected_put["timestamp"]

        rows.append(out)

    snapshot = pd.DataFrame(rows)

    snapshot["trade_dt"] = pd.to_datetime(snapshot["trade_dt"]).dt.normalize()
    snapshot["entry_tenor"] = snapshot["entry_tenor"].astype(int)

    snapshot.to_parquet(output_path, index=False)

    print("Saved intraday snapshot:")
    print(output_path)
    print("Rows:", len(snapshot))
    print("Status counts:")
    print(snapshot["intraday_status"].value_counts(dropna=False))

    return snapshot


# Test using EOD quote time, which exists.
intraday_test_snapshot = build_intraday_snapshot_from_cache(
    snapshot_trade_dt=latest_dt,
    snapshot_quote_time="160000",
)

display(
    intraday_test_snapshot[
        [
            "trade_dt",
            "entry_tenor",
            "expiry",
            "underlying_price",
            "atm_strike",
            "atm_put_bid",
            "atm_put_ask",
            "atm_put_mid",
            "intraday_status",
            "selected_chain_file_intraday",
        ]
    ]
)

Saved intraday snapshot:
C:\Users\patri\vrp_project\data\processed\naked_atm_put_intraday_snapshot_v0_1.parquet
Rows: 9
Status counts:
intraday_status
mapped    9
Name: count, dtype: int64


,trade_dt,entry_tenor,expiry,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,intraday_status,selected_chain_file_intraday
0,2026-06-25,9,2026-07-02,7357.49,7355.0,59.9,61.5,60.70,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl
1,2026-06-25,12,2026-07-10,7357.49,7355.0,84.8,86.2,85.50,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl
2,2026-06-25,15,2026-07-10,7357.49,7355.0,84.8,86.2,85.50,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl
3,2026-06-25,18,2026-07-10,7357.49,7355.0,84.8,86.2,85.50,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl
4,2026-06-25,21,2026-07-17,7357.49,7355.0,100.8,101.9,101.35,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl
5,2026-06-25,24,2026-07-17,7357.49,7355.0,100.8,101.9,101.35,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl
6,2026-06-25,27,2026-07-24,7357.49,7350.0,115.6,116.8,116.20,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260724_160000.pkl
7,2026-06-25,30,2026-07-24,7357.49,7350.0,115.6,116.8,116.20,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260724_160000.pkl
8,2026-06-25,33,2026-07-31,7357.49,7355.0,132.0,133.2,132.60,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260731_160000.pkl


In [42]:
# ============================================================
# Cell 23: Data update status summary
# ============================================================

eod_panel_current = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()
eod_panel_current["trade_dt"] = pd.to_datetime(eod_panel_current["trade_dt"]).dt.normalize()

status_summary = {
    "eod_panel_path": str(NAKED_ATM_EOD_PANEL),
    "eod_rows": len(eod_panel_current),
    "eod_start": eod_panel_current["trade_dt"].min(),
    "eod_end": eod_panel_current["trade_dt"].max(),
    "eod_unique_dates": eod_panel_current["trade_dt"].nunique(),
    "eod_unique_tenors": sorted(eod_panel_current["entry_tenor"].astype(int).unique().tolist()),
    "intraday_snapshot_path": str(NAKED_ATM_INTRADAY_SNAPSHOT),
    "theta_chain_cache_dir": str(THETADATA_CHAIN_DIR),
    "chain_cache_files": len(list(THETADATA_CHAIN_DIR.glob("*.pkl"))),
    "downloader_implemented": False,
    "ready_for_historical_cache_refresh": True,
    "ready_for_intraday_from_cached_files": True,
    "remaining_task": "Implement ThetaData downloader for new EOD/intraday chain files.",
}

display(pd.DataFrame([status_summary]))

,eod_panel_path,eod_rows,eod_start,eod_end,eod_unique_dates,eod_unique_tenors,intraday_snapshot_path,theta_chain_cache_dir,chain_cache_files,downloader_implemented,ready_for_historical_cache_refresh,ready_for_intraday_from_cached_files,remaining_task
0,C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet,18099,2018-06-25,2026-06-25,2011,"[9, 12, 15, 18, 21, 24, 27, 30, 33]",C:\Users\patri\vrp_project\data\processed\naked_atm_put_intraday_snapshot_v0_1.parquet,C:\Users\patri\vrp_project\data\raw\thetadata_chains,10901,False,True,True,Implement ThetaData downloader for new EOD/intraday chain files.
